# Práctica 1 · Autoencoder para Kaggle · v2 (Colab con GPU)





In [ ]:
# Prueba rápida: True solo para comprobar que todo funciona (pocas imágenes y 1 época)
QUICK_TEST = False

# torch se importa ANTES que TensorFlow 
import subprocess, sys
try:
    import torchmetrics
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics[image]'], check=True)
import torch
from torchmetrics.image.fid import FrechetInceptionDistance

import os, glob, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture

import tensorflow as tf
import keras
from keras import layers

SEED = 1234
keras.utils.set_random_seed(SEED)
GPU = len(tf.config.list_physical_devices('GPU')) > 0
if GPU:
    keras.mixed_precision.set_global_policy('mixed_float16')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPU:', GPU, '| FID en', DEVICE)

## 1. Datos



In [ ]:
# Si has subido los modelos de la v1 (modelos_ae_colab.zip), se descomprimen para no reentrenar
if os.path.exists('modelos_ae_colab.zip') and not os.path.isdir('modelos_ae_colab'):
    import zipfile
    zipfile.ZipFile('modelos_ae_colab.zip').extractall('modelos_ae_colab')

REPO = 'https://github.com/daaniii22/practica1-generativos.git'
if os.path.isdir('data/images'):
    DATA_DIR = 'data/images'
else:
    if not os.path.isdir('practica1-generativos'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--no-single-branch', REPO], check=True)
    DATA_DIR = 'practica1-generativos/data/images'

imgs_arr = []
for img_path in sorted(glob.glob(DATA_DIR + '/*.jpg')):
    imgs_arr.append(cv2.imread(img_path, cv2.IMREAD_GRAYSCALE))
x = np.array(imgs_arr)
if QUICK_TEST:
    x = x[:3000]
print(x.shape, x.dtype)

x = x.astype('float32') / 255.
x = np.reshape(x, (len(x), 64, 64, 1))
x_train, x_val = train_test_split(x, test_size=0.1, random_state=42)
print(x_train.shape, x_val.shape)

**Data augmentation:** durante el entrenamiento cada cara se voltea horizontalmente al azar. Una cara volteada sigue siendo una cara válida, así que es como tener el doble de datos.

In [ ]:
BATCH_SIZE = 128

def make_dataset(data, training):
    ds = tf.data.Dataset.from_tensor_slices(data)
    if training:
        ds = ds.shuffle(10000, seed=SEED).map(tf.image.random_flip_left_right, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda img: (img, img))
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(x_train, training=True)
val_ds = make_dataset(x_val, training=False)

## 2. Arquitectura

| Bloque | Capas | Salida |
|---|---|---|
| Encoder | 4 × [Conv2D 4×4 stride 2 + Conv2D 3×3], cada una con BatchNormalization + LeakyReLU | 64 → 32 → 16 → 8 → 4 (64, 128, 256 y 512 filtros) |
| Latente | Flatten + Dense(256) + BatchNormalization sin escala ni desplazamiento | 256 números con media ≈ 0 y varianza ≈ 1 |
| Decoder | Dense + Reshape(4, 4, 512) y 4 × [UpSampling2D + 2 × Conv2D 3×3] | 4 → 8 → 16 → 32 → 64 (512, 256, 128 y 64 filtros) |
| Salida | Conv2D(1, 3×3, sigmoid) | 64 × 64 × 1 en [0, 1] |

- **Latente estandarizado**: la `BatchNormalization` final del encoder obliga a que cada dimensión tenga media 0 y varianza 1, así que el espacio latente queda ordenado y se puede muestrear.
- El decoder usa **UpSampling2D + Conv2D**, la combinación del notebook 2_2_2, con dos convoluciones por resolución para pintar más detalle.

In [ ]:
LATENT_DIM = 256

def conv_block(x, filters, strides=1, kernel=3):
    x = layers.Conv2D(filters, kernel, strides=strides, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.LeakyReLU(0.2)(x)

# Encoder
encoder_inputs = keras.Input(shape=(64, 64, 1))
x = encoder_inputs
for filters in (64, 128, 256, 512):
    x = conv_block(x, filters, strides=2, kernel=4)
    x = conv_block(x, filters)
x = layers.Flatten()(x)
z = layers.Dense(LATENT_DIM, kernel_regularizer=keras.regularizers.l2(1e-5))(x)
z = layers.BatchNormalization(center=False, scale=False, name='latent')(z)
encoder = keras.Model(encoder_inputs, z, name='encoder')

# Decoder
decoder_inputs = keras.Input(shape=(LATENT_DIM,))
x = layers.Dense(4 * 4 * 512, use_bias=False)(decoder_inputs)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU(0.2)(x)
x = layers.Reshape((4, 4, 512))(x)
for filters in (512, 256, 128, 64):
    x = layers.UpSampling2D((2, 2), interpolation='bilinear')(x)
    x = conv_block(x, filters)
    x = conv_block(x, filters)
decoder_outputs = layers.Conv2D(1, 3, padding='same', activation='sigmoid', dtype='float32')(x)
decoder = keras.Model(decoder_inputs, decoder_outputs, name='decoder')

autoencoder = keras.Model(encoder_inputs, decoder(encoder(encoder_inputs)), name='autoencoder')
print(f'Encoder: {encoder.count_params():,} parámetros | Decoder: {decoder.count_params():,} parámetros')

## 3. Pérdida y entrenamiento

**Pérdida = 0,5 · MAE + 0,5 · (1 − SSIM)**
- **MAE** (error absoluto medio, notebook 2_3_1): hace que cada píxel se parezca al original. Emborrona menos que el MSE.
- **SSIM** (similitud estructural): compara bordes y texturas locales y premia que la estructura de la cara se conserve, que es lo que mide el FID. No es una arquitectura nueva, solo una forma de medir el error.

Callbacks (notebook 1_2_05):
- `ReduceLROnPlateau` reduce el learning rate a la mitad cuando la validación se estanca.
- `EarlyStopping` para si no mejora en 8 épocas y se queda con los mejores pesos.

In [ ]:
def mae_ssim_loss(y_true, y_pred):
    y_true = tf.cast(y_true, 'float32')
    y_pred = tf.cast(y_pred, 'float32')
    mae = tf.reduce_mean(tf.abs(y_true - y_pred))
    ssim = tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    return 0.5 * mae + 0.5 * (1.0 - ssim)

autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss=mae_ssim_loss)

EPOCHS = 1 if QUICK_TEST else 60
callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
]
CARGAR = os.path.exists('modelos_ae_colab/encoder.keras') and not QUICK_TEST
if CARGAR:
    encoder.set_weights(keras.models.load_model('modelos_ae_colab/encoder.keras', compile=False).get_weights())
    decoder.set_weights(keras.models.load_model('modelos_ae_colab/decoder.keras', compile=False).get_weights())
    logs = None
    print('Modelo v1 cargado: no se reentrena')
else:
    t0 = time.time()
    logs = autoencoder.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=2)
    print(f'Entrenamiento: {(time.time() - t0) / 60:.1f} min')

In [ ]:
if logs is not None:
    plt.figure(figsize=(8, 4))
    plt.plot(logs.history['loss'], label='train')
    plt.plot(logs.history['val_loss'], label='validación')
    plt.title('Pérdida (MAE + SSIM)')
    plt.xlabel('Época')
    plt.legend()
    plt.show()

decoded_imgs = autoencoder.predict(x_val[:10], verbose=0)
n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_val[i].reshape(64, 64), cmap='gray')
    ax.axis('off')
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(np.asarray(decoded_imgs[i]).reshape(64, 64), cmap='gray')
    ax.axis('off')
plt.show()

## 4. ¿Cómo generamos las caras? Elegimos midiendo el FID

Para generar caras nuevas necesitamos vectores latentes "creíbles". Comparamos varias formas de obtenerlos:

1. **N(0, I)**: números al azar de una normal estándar, como en un VAE.
2. **N(0, I) con temperatura 0,85**: lo mismo, pero más concentrado en el centro.
3. **Gaussiana ajustada**: una normal con la media y la covarianza reales de los vectores de entrenamiento.
4. **GMM** (*mezcla de gaussianas*, vista en *Introducción a los Métodos Generativos*) de 10, 30 y 60 componentes, ajustada a los vectores de entrenamiento. Cada componente modela una "zona" del espacio latente, por ejemplo un tipo de cara.
5. **GMM con temperatura τ** (0,9 y 0,8): se muestrea cada componente con su dispersión multiplicada por τ, es decir, más cerca de su centro. Es la idea del notebook `VAE_clase`: las caras del centro del espacio latente salen mejor que las de los bordes.

Medimos el FID de 1.000 caras generadas con cada método frente a las caras de validación y nos quedamos con el mejor.

In [ ]:
def a_uint8_3canales(imgs):
    imgs = np.asarray(imgs, dtype='float32')
    imgs = (np.clip(imgs, 0, 1) * 255).round().astype(np.uint8)
    if imgs.ndim == 3:
        imgs = imgs[..., None]
    return torch.from_numpy(np.repeat(imgs, 3, axis=-1).transpose(0, 3, 1, 2).copy())

N_REF = 300 if QUICK_TEST else 3000
fid_metric = FrechetInceptionDistance(feature=2048, reset_real_features=False).to(DEVICE)
reales = a_uint8_3canales(x_val[:N_REF])
with torch.inference_mode():
    for i in range(0, len(reales), 50):
        fid_metric.update(reales[i:i + 50].to(DEVICE), real=True)

import gc

def calcular_fid(imgs):
    fid_metric.reset()
    g = a_uint8_3canales(imgs)
    with torch.inference_mode():
        for i in range(0, len(g), 50):
            fid_metric.update(g[i:i + 50].to(DEVICE), real=False)
        valor = fid_metric.compute().item()
    del g
    gc.collect()
    return valor

def generar(zs):
    return np.asarray(decoder.predict(zs.astype('float32'), batch_size=250, verbose=0), dtype='float32')

In [ ]:
z_train = np.asarray(encoder.predict(x_train, batch_size=500, verbose=0), dtype='float64')
rng = np.random.default_rng(SEED)
N_GEN = 300 if QUICK_TEST else 1000

gmm10 = GaussianMixture(10, covariance_type='full', reg_covar=1e-4, random_state=SEED).fit(z_train)
gmm30 = GaussianMixture(30, covariance_type='full', reg_covar=1e-4, random_state=SEED).fit(z_train)
gmm60 = GaussianMixture(60, covariance_type='full', reg_covar=1e-4, random_state=SEED).fit(z_train)

def muestrear_gmm(gmm, n, tau=1.0):
    # Elige una componente por muestra y genera z ~ N(media_k, tau^2 * covarianza_k)
    comp = rng.choice(gmm.n_components, size=n, p=gmm.weights_)
    z = np.empty((n, LATENT_DIM))
    for k in range(gmm.n_components):
        idx = np.where(comp == k)[0]
        if len(idx):
            L = np.linalg.cholesky(gmm.covariances_[k] * tau ** 2)
            z[idx] = gmm.means_[k] + rng.normal(size=(len(idx), LATENT_DIM)) @ L.T
    return z

muestreadores = {
    'N(0, I)': lambda n: rng.normal(size=(n, LATENT_DIM)),
    'N(0, I) · temperatura 0,85': lambda n: 0.85 * rng.normal(size=(n, LATENT_DIM)),
    'Gaussiana ajustada': lambda n: rng.multivariate_normal(z_train.mean(0), np.cov(z_train, rowvar=False), size=n),
    'GMM 10 componentes': lambda n: muestrear_gmm(gmm10, n),
    'GMM 30 componentes': lambda n: muestrear_gmm(gmm30, n),
    'GMM 30 · temperatura 0,9': lambda n: muestrear_gmm(gmm30, n, 0.9),
    'GMM 60 componentes': lambda n: muestrear_gmm(gmm60, n),
    'GMM 60 · temperatura 0,9': lambda n: muestrear_gmm(gmm60, n, 0.9),
    'GMM 60 · temperatura 0,8': lambda n: muestrear_gmm(gmm60, n, 0.8),
}

resultados = {}
for nombre, muestrear in muestreadores.items():
    resultados[nombre] = calcular_fid(generar(muestrear(N_GEN)))
    print(f'{nombre:30s} FID = {resultados[nombre]:.1f}')
resultados['Reconstrucciones (referencia)'] = calcular_fid(autoencoder.predict(x_val[:N_GEN], verbose=0))
print(f"{'Reconstrucciones (referencia)':30s} FID = {resultados['Reconstrucciones (referencia)']:.1f}")

mejor = min(muestreadores, key=lambda k: resultados[k])
print('\nMejor forma de generar:', mejor)

In [ ]:
plt.figure(figsize=(10, 4))
nombres = list(resultados)
plt.barh(nombres, [resultados[k] for k in nombres], color=['#2F5FD0' if k == mejor else '#B8C2D6' for k in nombres])
plt.xlabel('FID (menor es mejor)')
plt.gca().invert_yaxis()
plt.show()

muestra = generar(muestreadores[mejor](20))
plt.figure(figsize=(20, 4.4))
plt.suptitle(f'Caras generadas · {mejor}', fontsize=15)
for i in range(20):
    ax = plt.subplot(2, 10, i + 1)
    plt.imshow(muestra[i].reshape(64, 64), cmap='gray')
    ax.axis('off')
plt.show()

## 5. Submission para Kaggle



In [ ]:
n_images = 200
rng = np.random.default_rng(2026)
caras_kaggle = generar(muestreadores[mejor](n_images))

out_folder = 'generated_imgs_ae_colab_v2'
os.makedirs(out_folder, exist_ok=True)
for i in range(n_images):
    img = (np.clip(caras_kaggle[i].reshape(64, 64), 0, 1) * 255).round().astype(np.uint8)
    Image.fromarray(img, 'L').save(f'{out_folder}/img_{i+1:03d}.png')

In [ ]:
out_folder = 'generated_imgs_ae_colab_v2'

# Load images in b&w 64x64 resolution
imgs_arr = []
for img_path in glob.glob(out_folder + '/*.png'):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)
    imgs_arr.append(img.flatten())
df = pd.DataFrame(np.array(imgs_arr))

# Add ID and Usage column
df.insert(0, 'Usage', 'Public')
df.insert(0, 'ID', range(1, len(df) + 1))
# Create .csv
df.to_csv('submission_ae_colab_v2.csv', index=False)
print(df.shape)

In [ ]:
def caras_csv(ruta):
    d = pd.read_csv(ruta)
    return d.iloc[:, 2:].values.reshape(-1, 64, 64, 1).astype('float32') / 255.

candidatas = {'Esta submission (AE Colab v2)': 'submission_ae_colab_v2.csv'}
if os.path.exists('submission_ae_colab.csv'):
    candidatas['AE Colab v1'] = 'submission_ae_colab.csv'
repo_dir = 'practica1-generativos' if os.path.isdir('practica1-generativos/.git') else '.'
for nombre, archivo in [('Dani · ae_complex (Kaggle 124,65)', 'submission_ae_complex.csv'),
                        ('Dani · ae_complex2', 'submission_ae_complex2.csv')]:
    r = subprocess.run(['git', '-C', repo_dir, 'show', f'origin/complex-ae-dani:{archivo}'], capture_output=True)
    if r.returncode == 0:
        with open(archivo, 'wb') as f:
            f.write(r.stdout)
        candidatas[nombre] = archivo

comparacion = {k: calcular_fid(caras_csv(v)) for k, v in candidatas.items()}
for k, v in sorted(comparacion.items(), key=lambda kv: kv[1]):
    print(f'{k:36s} FID = {v:.1f}')

## 7. Guardar y descargar

Se guardan el encoder y el decoder (`.keras`) y se descargan el CSV y los modelos. Fuera de Colab, los archivos quedan en la carpeta del notebook.

In [ ]:
if not CARGAR:
    os.makedirs('modelos_ae_colab', exist_ok=True)
    encoder.save('modelos_ae_colab/encoder.keras')
    decoder.save('modelos_ae_colab/decoder.keras')
pd.Series(resultados).to_csv('fid_muestreo_v2.csv')

try:
    from google.colab import files
    files.download('submission_ae_colab_v2.csv')
    files.download('fid_muestreo_v2.csv')
    if not CARGAR:
        import shutil
        shutil.make_archive('modelos_ae_colab', 'zip', 'modelos_ae_colab')
        files.download('modelos_ae_colab.zip')
except ImportError:
    print('Fuera de Colab: archivos guardados en la carpeta actual')